# SQL Query Agent in LangChain

Langchain provides a SQL Agent which offers a flexible way of interacting with SQL Databases. The SQL Agent system can answer questions intelligently by utilizing both the schema and content of the database, allowing it to provide insights into specific tables or records as needed. It can effectively handle and recover from errors by executing a generated query, catching any errors in the traceback, and adjusting the query accordingly. The system is optimized to query the database as many times as necessary to gather the information required to answer user questions fully. Additionally, it conserves resources by only retrieving the schema of relevant tables, ensuring efficient use of tokens and improving performance.


## Lab Description:

This lab explores how to use LangChain’s SQL Query Agent to interact with a SQLite database (chinook.db). Participants will learn two methods to fetch table names and retrieve table information before using the SQL Agent to execute queries. The lab demonstrates how the agent can automatically generate SQL queries, retrieve data, and provide table descriptions, making database interactions more efficient and intuitive.

## Lab Objectives

- Learn how to connect LangChain to a SQLite database (chinook.db).
- Explore methods to fetch table names and retrieve table information.
- Utilize LangChain’s SQL Query Agent to execute queries dynamically.
- Generate and interpret table descriptions using the SQL Agent.

## Load the LLM 

First step is to load the LLM. We use gemma2 9 billion parameter model. 

In [2]:
from langchain_ollama.llms import OllamaLLM

In [26]:
llm = OllamaLLM(model = "llama3.3:70b", base_url="http://10.79.253.112:11434", temperature=0)

## The Langchain SQL Database Toolkit

The `SQLDatabaseToolkit` contains tools that helps interact with a `SQL` database. `SQLDatabaseToolkit` enables SQL Agents to answer questions using data in a database. `.from_uri` method constructs a SQLAlchemy engine from URI. The URI string specifies the path to the database. The database should be located in the same directory as the notebook. We use SQLLite as it is lightweight and serverless. Working with agents are a lot easier when using lightweight SQL Engines. 

In [4]:
from langchain_community.utilities import SQLDatabase

## Chinook Database

Chinook is a sample database available for SQL Server, Oracle, MySQL, etc. It can be created by running a single SQL script. Chinook database is an alternative to the Northwind database, being ideal for demos and testing ORM tools targeting single and multiple database servers.

<div style="text-align: center;">
    <img src="db.png" alt="flow" width="720" height="480">
</div>

ref : https://github.com/lerocha/chinook-database

In [5]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

## `get_usable_table_names` Method

The `get_usable_table_names` returns the names of all tables in the database.

In [6]:
db.get_usable_table_names()

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

## `get_table_info` Method

`get_table_info` method can be called to fetch details about specific tables. 

In [7]:
print(db.get_table_info(["Album"]))


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


This returned a description about the Albums table. `get_table_info` accepts a list of strings so that description about multiple tables can be fetched. 

In [13]:
print(db.get_table_info(["Album", "Artist"]))


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


## The `create_sql_agent` Method

`create_sql_agent` is used to create a SQLAgent from an LLM and database. We provide the LLM, and the db object we initialized to the `create_sql_agent`method. 

In [27]:
from langchain_community.agent_toolkits import create_sql_agent
from langchain.agents import AgentType

In [28]:
agent = create_sql_agent(llm, db=db, handle_parsing_errors=True, verbose = True)

Now that we have initialized the agent, we can invoke it with the `.invoke` method. 

In [29]:
result = agent.invoke("How many different Artists are in the database?")



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
With the list of tables, I can see that the "Artist" table is likely to be the most relevant for answering this question. Next, I should query the schema of the "Artist" table to confirm what columns are available.

Action: sql_db_schema
Action Input: Artist
CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
With the schema of the "Artist" table, I can see that it has two columns: "ArtistId" and "Name". To find out how many different artists are in the database, I should query the "Artist" table to count the number of unique rows.

Action: sql_db_query_checker
```sqlT COUNT(*) FROM Artist
SELECT COUNT(*) FROM Artist;
The query looks correct. Now I can execute it.

Action: sql_db_query
I now know the final answert[(275,)]
Final Answer: There are 275 different Artists in the database.

> Fi

In [30]:
result

{'input': 'How many different Artists are in the database?',
 'output': 'There are 275 different Artists in the database.'}

The agent generated a response by querying the database. 

We can set `verbose=True` to see the step by step thought process of the agent. 

In [31]:
agent = create_sql_agent(llm, db=db, verbose=True, handle_parsing_errors=True)

In [32]:
agent.invoke("List the total sales per country")



> Entering new SQL Agent Executor chain...
To find the total sales per country, we first need to identify which tables in the database might contain information about sales and countries.

Action: sql_db_list_tables
With the list of tables, I can now identify which ones might be relevant to sales and countries. The "Invoice" table seems like a good candidate for sales information, and possibly the "Customer" table for country information.

Action: sql_db_schema
Action Input: Invoice, Customer
CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows 

{'input': 'List the total sales per country',
 'output': 'The total sales per country are as follows: \n- USA: $523.06\n- Canada: $303.96\n- France: $195.10\n- Brazil: $190.10\n- Germany: $156.48\n- United Kingdom: $112.86\n- Czech Republic: $90.24\n- Portugal: $77.24\n- India: $75.26\n- Chile: $46.62'}

The agent identifies the relevant table based on the user query. It then analyses the schema of the identified table to generate a syntactically correct query to generate the final output. 

## Table Descriptions

SQLAgents can also describe the content of a specific table. That is, it can provide all information on what the table is about. 

In [33]:
agent.invoke("What is the 'Employee' table about ?")



> Entering new SQL Agent Executor chain...
To find out more about the 'Employee' table, first, we need to confirm that such a table exists and then examine its structure.

Action: sql_db_list_tables
The 'Employee' table is present in the database. Now that I have confirmed its existence, I should query the schema of this table to understand what it's about.

Action: sql_db_schema
Action Input: Employee
CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" DATETIME, 
	"HireDate" DATETIME, 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60), 
	PRIMARY KEY ("EmployeeId"), 
	FOREIGN KEY("ReportsTo") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Employee table:
EmployeeId	LastName	FirstName	Title	Repor

{'input': "What is the 'Employee' table about ?",
 'output': "The 'Employee' table stores information about employees, including their first name, last name, and job title. Some examples of employee titles include General Manager, Sales Manager, Sales Support Agent, IT Manager, and IT Staff. Specifically, some employees in the database are Andrew Adams (General Manager), Nancy Edwards (Sales Manager), Jane Peacock (Sales Support Agent), Margaret Park (Sales Support Agent), Steve Johnson (Sales Support Agent), Michael Mitchell (IT Manager), Robert King (IT Staff), and Laura Callahan (IT Staff)."}

The agent provided a description about the Employee table. 

<div style="text-align: left;">
    <img src="logo.png" alt="flow" width="150" height="100">
</div>